# 🧠 Price Direction Prediction using Machine Learning
## Trading Journal AI Module — University Thesis

**Author**: [Your Name]

This notebook trains and compares **3 ML models** for next-day price direction prediction:
1. **LSTM** (Long Short-Term Memory Neural Network)
2. **Random Forest** (Bagging Ensemble)
3. **XGBoost** (Gradient Boosting)

**Data Source**: Yahoo Finance (via `yfinance`)

---

### ⚡ How to Run
1. Click **Runtime → Run all** (or run cells one by one)
2. Training takes ~5-10 min with GPU
3. All figures are saved + downloadable at the end

### 🔧 GPU Setup
Go to **Runtime → Change runtime type → GPU (T4)**

---
## 1. Setup & Install Dependencies

In [ ]:
# Install required packages (Colab has TF pre-installed)
!pip install -q yfinance ta xgboost shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

from xgboost import XGBClassifier
import yfinance as yf
from ta import trend, momentum, volatility, volume
import shap
import joblib
import json
from datetime import datetime, timedelta
from pathlib import Path

# Plotting config
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 150

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Output directory
SAVE_DIR = Path('saved_models')
FIGURES_DIR = SAVE_DIR / 'figures'
SAVE_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')
print('✅ Setup complete!')

---
## 2. Configuration

Change `SYMBOL` to train on a different instrument.

In [ ]:
# ============================================
# ⚙️ CONFIGURATION — EDIT THESE VALUES
# ============================================
SYMBOL = 'EURUSD=X'      # Trading symbol (AAPL, BTC-USD, GBPUSD=X, etc.)
YEARS = 5                 # Years of historical data
SEQUENCE_LENGTH = 60      # LSTM lookback window (trading days)
TRAIN_RATIO = 0.70        # 70% training
VAL_RATIO = 0.15          # 15% validation
TEST_RATIO = 0.15         # 15% testing

# LSTM hyperparameters
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 15             # Early stopping patience

# Set to False for faster training (skips GridSearchCV)
TUNE_HYPERPARAMS = True

print(f'Training config: {SYMBOL}, {YEARS} years, sequence={SEQUENCE_LENGTH}')

---
## 3. Data Collection

**Data Source**: Yahoo Finance via `yfinance` library.

**Citation**: Yahoo Finance historical market data, accessed via yfinance (Aroussi, R., 2024).

We fetch **OHLCV** (Open, High, Low, Close, Volume) daily data, auto-adjusted for splits and dividends.

In [ ]:
# Fetch historical data
end_date = datetime.now()
start_date = end_date - timedelta(days=YEARS * 365)

print(f'Fetching {SYMBOL} data from {start_date.date()} to {end_date.date()}...')

ticker = yf.Ticker(SYMBOL)
raw_df = ticker.history(
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d'),
    interval='1d',
    auto_adjust=True,
)

# Clean up
raw_df = raw_df.drop(columns=['Dividends', 'Stock Splits'], errors='ignore')
raw_df.index = pd.to_datetime(raw_df.index).tz_localize(None)
raw_df = raw_df.sort_index().ffill().dropna()

print(f'\n📊 Dataset Summary:')
print(f'   Symbol:     {SYMBOL}')
print(f'   Period:     {raw_df.index[0].date()} → {raw_df.index[-1].date()}')
print(f'   Total rows: {len(raw_df)}')
print(f'   Columns:    {list(raw_df.columns)}')

raw_df.head()

In [ ]:
# Dataset statistics
print('📈 Descriptive Statistics:')
raw_df.describe()

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price chart
axes[0].plot(raw_df.index, raw_df['Close'], color='#2563eb', linewidth=1)
axes[0].set_title(f'{SYMBOL} — Closing Price ({raw_df.index[0].year}–{raw_df.index[-1].year})')
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(raw_df.index, raw_df['Volume'], color='#94a3b8', alpha=0.7, width=1)
axes[1].set_title('Trading Volume')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

# Returns distribution
returns = raw_df['Close'].pct_change().dropna()
axes[2].hist(returns, bins=100, color='#2563eb', alpha=0.7, edgecolor='white')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title(f'Daily Returns Distribution (μ={returns.mean():.5f}, σ={returns.std():.5f})')
axes[2].set_xlabel('Daily Return')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_overview.png', bbox_inches='tight')
plt.show()

print(f'UP days:   {(returns > 0).sum()} ({(returns > 0).mean():.1%})')
print(f'DOWN days: {(returns <= 0).sum()} ({(returns <= 0).mean():.1%})')
print(f'Skewness:  {returns.skew():.4f}')
print(f'Kurtosis:  {returns.kurtosis():.4f}')

---
## 5. Feature Engineering

We compute **15+ technical indicators** as input features.

| Category | Indicator | Description | Reference |
|----------|-----------|-------------|----------|
| Trend | SMA (5,10,20,50,200) | Simple Moving Average | — |
| Trend | EMA (12,26) | Exponential Moving Average | — |
| Trend | MACD + Signal + Hist | Moving Average Convergence Divergence | Appel (2005) |
| Trend | ADX (14) | Average Directional Index | Wilder (1978) |
| Momentum | RSI (14) | Relative Strength Index | Wilder (1978) |
| Momentum | Stochastic %K, %D | Stochastic Oscillator | Lane (1984) |
| Momentum | Williams %R (14) | Williams Percent Range | Williams (1979) |
| Momentum | ROC (10) | Rate of Change | — |
| Volatility | Bollinger Bands (20,2σ) | Price envelope | Bollinger (2002) |
| Volatility | ATR (14) | Average True Range | Wilder (1978) |
| Volume | OBV | On-Balance Volume | Granville (1963) |
| Volume | CCI (20) | Commodity Channel Index | Lambert (1980) |

In [ ]:
df = raw_df.copy()

# ========== TECHNICAL INDICATORS ==========

# --- Moving Averages ---
for period in [5, 10, 20, 50, 200]:
    df[f'sma_{period}'] = df['Close'].rolling(window=period).mean()
for period in [12, 26]:
    df[f'ema_{period}'] = df['Close'].ewm(span=period, adjust=False).mean()

# --- RSI (Wilder, 1978) ---
df['rsi_14'] = momentum.RSIIndicator(close=df['Close'], window=14).rsi()

# --- MACD (Appel, 2005) ---
macd_ind = trend.MACD(close=df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['macd'] = macd_ind.macd()
df['macd_signal'] = macd_ind.macd_signal()
df['macd_hist'] = macd_ind.macd_diff()

# --- Bollinger Bands (Bollinger, 2002) ---
bb = volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['bb_upper'] = bb.bollinger_hband()
df['bb_middle'] = bb.bollinger_mavg()
df['bb_lower'] = bb.bollinger_lband()
df['bb_width'] = bb.bollinger_wband()
df['bb_pct'] = bb.bollinger_pband()

# --- ATR (Wilder, 1978) ---
df['atr_14'] = volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# --- Stochastic Oscillator (Lane, 1984) ---
stoch = momentum.StochasticOscillator(high=df['High'], low=df['Low'], close=df['Close'], window=14, smooth_window=3)
df['stoch_k'] = stoch.stoch()
df['stoch_d'] = stoch.stoch_signal()

# --- OBV (Granville, 1963) ---
df['obv'] = volume.OnBalanceVolumeIndicator(close=df['Close'], volume=df['Volume']).on_balance_volume()

# --- ADX (Wilder, 1978) ---
adx_ind = trend.ADXIndicator(high=df['High'], low=df['Low'], close=df['Close'], window=14)
df['adx_14'] = adx_ind.adx()
df['di_plus'] = adx_ind.adx_pos()
df['di_minus'] = adx_ind.adx_neg()

# --- CCI (Lambert, 1980) ---
df['cci_20'] = trend.CCIIndicator(high=df['High'], low=df['Low'], close=df['Close'], window=20).cci()

# --- Williams %R (Williams, 1979) ---
df['willr_14'] = momentum.WilliamsRIndicator(high=df['High'], low=df['Low'], close=df['Close'], lbp=14).williams_r()

# --- ROC ---
df['roc_10'] = momentum.ROCIndicator(close=df['Close'], window=10).roc()

# ========== DERIVED FEATURES ==========

# Price returns
df['return_1d'] = df['Close'].pct_change(1)
df['return_5d'] = df['Close'].pct_change(5)
df['return_10d'] = df['Close'].pct_change(10)

# Volatility
df['volatility_10d'] = df['return_1d'].rolling(10).std()
df['volatility_20d'] = df['return_1d'].rolling(20).std()

# SMA crossover signals
df['sma_cross_5_20'] = (df['sma_5'] > df['sma_20']).astype(int)
df['sma_cross_50_200'] = (df['sma_50'] > df['sma_200']).astype(int)

# Bollinger Band distance (normalized)
bb_range = df['bb_upper'] - df['bb_lower']
df['bb_distance'] = (df['Close'] - df['bb_middle']) / bb_range.replace(0, np.nan)

# Price vs SMA50
df['price_vs_sma50'] = (df['Close'] - df['sma_50']) / df['sma_50']

# Volume change
df['volume_change'] = df['Volume'].pct_change(1).replace([np.inf, -np.inf], 0)

# High-Low range
df['high_low_range'] = (df['High'] - df['Low']) / df['Close']

# Day of week (cyclical encoding)
df['day_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 5)
df['day_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 5)

# ========== TARGET VARIABLE ==========
# 1 = tomorrow's close > today's close (UP)
# 0 = tomorrow's close <= today's close (DOWN)
df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Drop NaN rows (indicator warmup + last row from target shift)
rows_before = len(df)
df = df.dropna()

# Define feature columns (everything except OHLCV and target)
exclude = ['Open', 'High', 'Low', 'Close', 'Volume', 'target']
feature_columns = [col for col in df.columns if col not in exclude]

print(f'\n📊 Feature Engineering Complete:')
print(f'   Rows: {rows_before} → {len(df)} (dropped {rows_before - len(df)} warmup rows)')
print(f'   Features: {len(feature_columns)}')
print(f'   Target balance: UP={df["target"].mean():.1%}, DOWN={1-df["target"].mean():.1%}')
print(f'\n   Feature list:')
for i, col in enumerate(feature_columns, 1):
    print(f'   {i:2d}. {col}')

In [ ]:
# Class balance visualization
up = df['target'].sum()
down = len(df) - up

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([up, down], labels=[f'UP ({up})', f'DOWN ({down})'],
       colors=['#16a34a', '#dc2626'], autopct='%1.1f%%', startangle=90,
       textprops={'fontsize': 14})
ax.set_title('Target Variable Distribution')
fig.savefig(FIGURES_DIR / 'class_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
key_features = feature_columns[:25]  # Limit for readability
corr = df[key_features + ['target']].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## 6. Data Preprocessing

### Time-Series Split (NO Random Shuffle)

**Critical for academic rigor**: We use a **chronological split** to prevent **look-ahead bias**.
The model never sees future data during training.

```
|◄────── Train (70%) ──────►|◄── Val (15%) ──►|◄── Test (15%) ──►|
|     oldest data            |    middle         |    newest (future)|
```

### Feature Scaling
- **MinMaxScaler** fitted **ONLY on training data** → prevents data leakage
- Applied to validation and test sets using the training-fitted scaler

In [ ]:
# ========== TIME-SERIES SPLIT ==========
n = len(df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print('📊 Data Split (Time-Based, No Shuffle):')
print(f'   Train: {len(train_df):>5} rows  ({train_df.index[0].date()} → {train_df.index[-1].date()})')
print(f'   Val:   {len(val_df):>5} rows  ({val_df.index[0].date()} → {val_df.index[-1].date()})')
print(f'   Test:  {len(test_df):>5} rows  ({test_df.index[0].date()} → {test_df.index[-1].date()})')
print(f'   Total: {len(df):>5} rows')

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df.index, train_df['Close'], color='#2563eb', label=f'Train ({len(train_df)})', linewidth=1)
ax.plot(val_df.index, val_df['Close'], color='#f59e0b', label=f'Validation ({len(val_df)})', linewidth=1)
ax.plot(test_df.index, test_df['Close'], color='#dc2626', label=f'Test ({len(test_df)})', linewidth=1)
ax.axvline(x=train_df.index[-1], color='white', linestyle='--', alpha=0.3)
ax.axvline(x=val_df.index[-1], color='white', linestyle='--', alpha=0.3)
ax.set_title(f'{SYMBOL} — Train / Validation / Test Split')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'train_val_test_split.png', bbox_inches='tight')
plt.show()

In [ ]:
# ========== FEATURE SCALING ==========
# MinMaxScaler fitted on TRAINING data ONLY (prevents data leakage)
scaler = MinMaxScaler()

train_df[feature_columns] = scaler.fit_transform(train_df[feature_columns])
val_df[feature_columns] = scaler.transform(val_df[feature_columns])
test_df[feature_columns] = scaler.transform(test_df[feature_columns])

# Save scaler
joblib.dump(scaler, SAVE_DIR / 'scaler.pkl')
with open(SAVE_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_columns, f)

print('✅ Features scaled (MinMaxScaler fitted on training data only)')

# ========== PREPARE MODEL INPUTS ==========

# LSTM: 3D sequences (samples, timesteps, features)
def create_sequences(data, seq_len):
    X, y = [], []
    features = data[feature_columns].values
    target = data['target'].values
    for i in range(seq_len, len(features)):
        X.append(features[i-seq_len:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(train_df, SEQUENCE_LENGTH)
X_val_seq, y_val_seq = create_sequences(val_df, SEQUENCE_LENGTH)
X_test_seq, y_test_seq = create_sequences(test_df, SEQUENCE_LENGTH)

# Tree models: 2D flat (samples, features)
X_train_flat = train_df[feature_columns].values
y_train_flat = train_df['target'].values
X_val_flat = val_df[feature_columns].values
y_val_flat = val_df['target'].values
X_test_flat = test_df[feature_columns].values
y_test_flat = test_df['target'].values

n_features = len(feature_columns)
print(f'\n📊 Model Input Shapes:')
print(f'   LSTM:  X_train={X_train_seq.shape}  (samples × {SEQUENCE_LENGTH} timesteps × {n_features} features)')
print(f'   Flat:  X_train={X_train_flat.shape}  (samples × {n_features} features)')

---
## 7. Model 1: LSTM (Long Short-Term Memory)

### Theory

LSTMs are a type of Recurrent Neural Network (RNN) that solve the **vanishing gradient problem**
through a gating mechanism:

- **Forget Gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ — decides what to forget
- **Input Gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ — decides what to store
- **Output Gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ — decides what to output

**Architecture**:
```
Input (60 × N) → LSTM(128) → Dropout(0.2) → LSTM(64) → Dropout(0.2) → Dense(32, ReLU) → Dense(1, Sigmoid)
```

**Reference**: Hochreiter, S. & Schmidhuber, J. (1997). Long Short-Term Memory. Neural Computation, 9(8), 1735-1780.

In [ ]:
# Build LSTM
lstm_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, n_features)),
    LSTM(LSTM_UNITS_1, return_sequences=True),
    Dropout(DROPOUT_RATE),
    LSTM(LSTM_UNITS_2, return_sequences=False),
    Dropout(DROPOUT_RATE),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])

lstm_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

lstm_model.summary()

In [ ]:
# Train LSTM
print('🧠 Training LSTM...')
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ],
    verbose=1,
)

# Save model
lstm_model.save(SAVE_DIR / 'lstm_model.h5')
print(f'\n✅ LSTM trained ({len(history.history["loss"])} epochs)')
print(f'   Best val_loss: {min(history.history["val_loss"]):.4f}')
print(f'   Best val_acc:  {max(history.history["val_accuracy"]):.4f}')

In [ ]:
# Plot LSTM training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['loss']) + 1)

ax1.plot(epochs_range, history.history['loss'], 'b-', label='Training', linewidth=2)
ax1.plot(epochs_range, history.history['val_loss'], 'r-', label='Validation', linewidth=2)
ax1.set_title('LSTM — Loss Curves')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Binary Cross-Entropy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history.history['accuracy'], 'b-', label='Training', linewidth=2)
ax2.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
ax2.set_title('LSTM — Accuracy Curves')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'lstm_training.png', bbox_inches='tight')
plt.show()

---
## 8. Model 2: Random Forest

### Theory

Random Forest constructs multiple decision trees via **Bootstrap Aggregation** (Bagging).
Each tree is trained on a random subset of data and features, then predictions are
aggregated by majority vote.

**Hyperparameter tuning**: GridSearchCV with **TimeSeriesSplit** (respects temporal order).

**Reference**: Breiman, L. (2001). Random Forests. Machine Learning, 45(1), 5-32.

In [ ]:
print('🌲 Training Random Forest...')

if TUNE_HYPERPARAMS:
    param_grid = {
        'n_estimators': [100, 200, 500],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
    }
    cv = TimeSeriesSplit(n_splits=5)
    grid = GridSearchCV(
        RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
        param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1,
    )
    grid.fit(X_train_flat, y_train_flat)
    rf_model = grid.best_estimator_
    print(f'\n   Best params: {grid.best_params_}')
    print(f'   Best CV accuracy: {grid.best_score_:.4f}')
else:
    rf_model = RandomForestClassifier(
        n_estimators=200, max_depth=20, random_state=42,
        class_weight='balanced', n_jobs=-1,
    )
    rf_model.fit(X_train_flat, y_train_flat)

# Save
joblib.dump(rf_model, SAVE_DIR / 'rf_model.pkl')
print(f'\n✅ Random Forest trained')
print(f'   Train accuracy: {rf_model.score(X_train_flat, y_train_flat):.4f}')
print(f'   Val accuracy:   {rf_model.score(X_val_flat, y_val_flat):.4f}')

In [ ]:
# Feature importance (Random Forest)
rf_importance = dict(sorted(
    zip(feature_columns, rf_model.feature_importances_),
    key=lambda x: x[1], reverse=True
))

top_n = 15
top_feats = list(rf_importance.items())[:top_n][::-1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([f[0] for f in top_feats], [f[1] for f in top_feats], color='#2563eb', alpha=0.85)
ax.set_xlabel('Gini Importance')
ax.set_title(f'Random Forest — Top {top_n} Feature Importance')
ax.grid(True, axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'rf_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 9. Model 3: XGBoost

### Theory

XGBoost builds trees **sequentially** — each tree corrects the errors of the previous
ensemble by following the **negative gradient** of the loss function.

**Key innovations**: L1/L2 regularization, column subsampling, sparsity-aware splits.

**SHAP**: We use SHapley Additive exPlanations for model interpretability.

**References**:
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. ACM SIGKDD.
- Lundberg, S.M. & Lee, S.I. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.

In [ ]:
print('🚀 Training XGBoost...')

if TUNE_HYPERPARAMS:
    param_grid = {
        'n_estimators': [100, 300, 500],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    }
    cv = TimeSeriesSplit(n_splits=5)
    grid = GridSearchCV(
        XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False, n_jobs=-1, verbosity=0),
        param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1,
    )
    grid.fit(X_train_flat, y_train_flat)
    xgb_model = grid.best_estimator_
    print(f'\n   Best params: {grid.best_params_}')
    print(f'   Best CV accuracy: {grid.best_score_:.4f}')
else:
    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        random_state=42, eval_metric='logloss', use_label_encoder=False, n_jobs=-1,
    )
    xgb_model.fit(X_train_flat, y_train_flat, eval_set=[(X_val_flat, y_val_flat)], verbose=False)

# Save
xgb_model.save_model(str(SAVE_DIR / 'xgb_model.json'))
print(f'\n✅ XGBoost trained')
print(f'   Train accuracy: {xgb_model.score(X_train_flat, y_train_flat):.4f}')
print(f'   Val accuracy:   {xgb_model.score(X_val_flat, y_val_flat):.4f}')

In [ ]:
# XGBoost feature importance
xgb_importance = dict(sorted(
    zip(feature_columns, xgb_model.feature_importances_),
    key=lambda x: x[1], reverse=True
))

top_feats_xgb = list(xgb_importance.items())[:top_n][::-1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([f[0] for f in top_feats_xgb], [f[1] for f in top_feats_xgb], color='#dc2626', alpha=0.85)
ax.set_xlabel('Gain Importance')
ax.set_title(f'XGBoost — Top {top_n} Feature Importance')
ax.grid(True, axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'xgb_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP analysis (XGBoost explainability)
print('Computing SHAP values...')
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_flat[:200])

fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_flat[:200], feature_names=feature_columns, show=False, max_display=15)
plt.title('SHAP Summary — XGBoost Feature Contributions')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10. Model Comparison & Evaluation

### Metrics
- **Accuracy**: % of correct predictions
- **Precision**: Of predicted UPs, how many were correct
- **Recall**: Of actual UPs, how many were caught
- **F1 Score**: Harmonic mean of Precision and Recall
- **AUC-ROC**: Area under the ROC curve

In [ ]:
# Generate test predictions (probabilities)
lstm_proba = lstm_model.predict(X_test_seq, verbose=0).flatten()
rf_proba = rf_model.predict_proba(X_test_flat)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test_flat)[:, 1]

# Evaluate each model
def evaluate(y_true, y_proba, name):
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        'Model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_true, y_proba),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }

results = {
    'LSTM': evaluate(y_test_seq, lstm_proba, 'LSTM'),
    'Random Forest': evaluate(y_test_flat, rf_proba, 'Random Forest'),
    'XGBoost': evaluate(y_test_flat, xgb_proba, 'XGBoost'),
}

# Comparison table
comparison = pd.DataFrame([
    {k: f'{v:.4f}' if isinstance(v, float) else v
     for k, v in r.items() if k != 'confusion_matrix'}
    for r in results.values()
])

print('\n📊 MODEL COMPARISON (Test Set):')
comparison

In [ ]:
# Confusion Matrices (side-by-side)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, r) in zip(axes, results.items()):
    sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'], ax=ax)
    ax.set_title(f'{name}\nAcc={r["accuracy"]:.3f} F1={r["f1_score"]:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

fig.suptitle('Confusion Matrices — Model Comparison', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves (overlaid)
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2563eb', '#16a34a', '#dc2626']
min_len = min(len(lstm_proba), len(rf_proba))

for (name, proba, y_true), color in zip(
    [('LSTM', lstm_proba[:min_len], y_test_seq[:min_len]),
     ('Random Forest', rf_proba[:min_len], y_test_flat[:min_len]),
     ('XGBoost', xgb_proba[:min_len], y_test_flat[:min_len])],
    colors
):
    fpr, tpr, _ = roc_curve(y_true, proba)
    auc = roc_auc_score(y_true, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2)

ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
fig.savefig(FIGURES_DIR / 'roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Performance comparison bar chart
metrics_list = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
labels = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, r) in enumerate(results.items()):
    vals = [r[m] for m in metrics_list]
    ax.bar(x + i*width, vals, width, label=name, color=colors[i], alpha=0.85)

ax.set_ylabel('Score'); ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width); ax.set_xticklabels(labels)
ax.legend(); ax.grid(True, axis='y', alpha=0.3); ax.set_ylim(0.4, 0.75)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()

---
## 11. Backtesting

Simulating a **long-only strategy**:
- Model predicts UP → Buy
- Model predicts DOWN → Stay in cash

**Parameters**: $10,000 initial capital, 0.1% commission per trade.

**Baseline**: Buy-and-hold (buy day 1, hold until end).

In [ ]:
INITIAL_CAPITAL = 10000
COMMISSION = 0.001

def backtest(prices, predictions, threshold=0.5):
    n = len(prices)
    signals = (predictions >= threshold).astype(int)
    equity = np.zeros(n)
    equity[0] = INITIAL_CAPITAL
    position = 0
    trades = 0

    for i in range(1, n):
        daily_ret = (prices[i] - prices[i-1]) / prices[i-1]
        if signals[i] != position:
            trades += 1
            position = signals[i]
            equity[i] = equity[i-1] * (1 - COMMISSION)
        else:
            equity[i] = equity[i-1]
        if position == 1:
            equity[i] *= (1 + daily_ret)

    total_return = (equity[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
    daily_rets = np.diff(equity) / equity[:-1]
    sharpe = np.mean(daily_rets) / (np.std(daily_rets) + 1e-8) * np.sqrt(252)
    peak = np.maximum.accumulate(equity)
    max_dd = np.max((peak - equity) / peak) * 100

    return {'equity': equity, 'return': total_return, 'sharpe': sharpe,
            'max_dd': max_dd, 'trades': trades}

# Test prices
test_prices = test_df['Close'].values

# Buy & Hold baseline
bh_equity = INITIAL_CAPITAL * (test_prices / test_prices[0])
bh_return = (bh_equity[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100

# Backtest each model
bt_lstm = backtest(test_prices[len(test_prices)-len(lstm_proba):], lstm_proba)
bt_rf = backtest(test_prices, rf_proba)
bt_xgb = backtest(test_prices, xgb_proba)

# Results table
bt_table = pd.DataFrame([
    {'Strategy': 'Buy & Hold', 'Return (%)': f'{bh_return:.2f}', 'Sharpe': '—', 'Max DD (%)': '—', 'Trades': 1},
    {'Strategy': 'LSTM', 'Return (%)': f'{bt_lstm["return"]:.2f}', 'Sharpe': f'{bt_lstm["sharpe"]:.2f}', 'Max DD (%)': f'{bt_lstm["max_dd"]:.2f}', 'Trades': bt_lstm['trades']},
    {'Strategy': 'Random Forest', 'Return (%)': f'{bt_rf["return"]:.2f}', 'Sharpe': f'{bt_rf["sharpe"]:.2f}', 'Max DD (%)': f'{bt_rf["max_dd"]:.2f}', 'Trades': bt_rf['trades']},
    {'Strategy': 'XGBoost', 'Return (%)': f'{bt_xgb["return"]:.2f}', 'Sharpe': f'{bt_xgb["sharpe"]:.2f}', 'Max DD (%)': f'{bt_xgb["max_dd"]:.2f}', 'Trades': bt_xgb['trades']},
])

print('\n📊 Backtesting Results:')
bt_table

In [ ]:
# Equity curves
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(test_df.index, bh_equity, '--', color='#94a3b8', label=f'Buy & Hold ({bh_return:+.1f}%)', linewidth=2)
ax.plot(test_df.index[-len(bt_lstm["equity"]):], bt_lstm['equity'], color='#2563eb', label=f'LSTM ({bt_lstm["return"]:+.1f}%)', linewidth=2)
ax.plot(test_df.index, bt_rf['equity'], color='#16a34a', label=f'Random Forest ({bt_rf["return"]:+.1f}%)', linewidth=2)
ax.plot(test_df.index, bt_xgb['equity'], color='#dc2626', label=f'XGBoost ({bt_xgb["return"]:+.1f}%)', linewidth=2)

ax.set_xlabel('Date'); ax.set_ylabel('Portfolio Value ($)')
ax.set_title('Backtesting — Equity Curves')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'equity_curves.png', bbox_inches='tight')
plt.show()

---
## 12. Save Results & Download

All trained models and figures are saved to `saved_models/`.

In [ ]:
# Save metrics
metrics_export = {}
for name, r in results.items():
    metrics_export[name] = {k: float(v) if isinstance(v, (float, np.floating)) else v
                            for k, v in r.items() if k not in ['confusion_matrix', 'Model']}

with open(SAVE_DIR / 'training_metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

print('='*60)
print('✅ ALL TRAINING COMPLETE')
print('='*60)
print(f'\nFiles saved to {SAVE_DIR}/')
for f in sorted(SAVE_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        unit = 'KB' if size < 1_000_000 else 'MB'
        val = size/1024 if size < 1_000_000 else size/1_000_000
        print(f'   {f.relative_to(SAVE_DIR)} ({val:.1f} {unit})')

In [ ]:
# Download models (Colab only)
try:
    from google.colab import files
    import shutil
    shutil.make_archive('saved_models', 'zip', '.', 'saved_models')
    files.download('saved_models.zip')
    print('📥 Download started! Save this zip and extract to backend/ai/saved_models/')
except:
    print('Not running in Colab — models saved locally.')

---
## 13. Conclusions

*(Fill in after reviewing results)*

### Summary
- Three ML models were trained for next-day price direction prediction of {SYMBOL}
- Features: {N} technical indicators derived from 5 years of OHLCV data
- Best model: [Model] with accuracy of X% and F1 of Y

### Limitations
- Efficient Market Hypothesis suggests limited predictability from historical prices
- Transaction costs reduce real-world returns
- Model performance varies across market regimes

### Future Work
- Incorporate sentiment analysis
- Test Transformer/Attention architectures
- Multi-timeframe analysis